# Becke Partition 二阶梯度代码迁移准备

本 notebook 将 `10-3-becke_partition_deriv2.ipynb` 的二阶解析格点权重导数 $\partial^2 w_g / \partial R_{At} \partial R_{Bs}$，从**全张量向量化**实现迁移为 `10-4-becke_rsprep_deriv1.ipynb` 的**按格点 batch 循环 (rsprep)** 实现，作为后续迁移到 C/Rust 的中间算法。

实现策略与 `10-4` 保持一致：以格点 batch 为外层循环，避免 `10-3` 中 $(\mathrm{natm}, \mathrm{natm}, 3, \mathrm{natm}, 3, n_\mathrm{batch})$ 量级的 6 维中间张量 (如 `ddR_log_P`、`ddR_P`)；二阶对数导数 (L2) 的贡献在 per-pair 循环中直接累加进 5 维输出张量。

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial
import scipy

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz_orig = np.array([[0.0, 0.0, 0.0], [1.0, 0.1, 0.2], [0.3, 1.1, 0.2], [0.1, 0.1, 1.2]])  # in angstrom

def perturb_xyz(xyz, A, t, sgn, delta):
    xyz_pert = xyz.copy()
    xyz_pert[A, t] += sgn * delta
    return xyz_pert

def perturb_mol(A, t, sgn, delta):
    xyz_coords = perturb_xyz(xyz_orig, A, t, sgn, delta)
    atm_symbols = ["N", "H", "H", "H"]
    xyz_str = "\n".join(f"{atm} {x:.6f} {y:.6f} {z:.6f}" for atm, (x, y, z) in zip(atm_symbols, xyz_coords))
    mol_pert = gto.Mole(atom=xyz_str, basis="def2-TZVP", max_memory=32000).build()
    return mol_pert

def perturb_mol_grids(A, t, sgn, delta):
    mol_pert = perturb_mol(A, t, sgn, delta)
    grids_pert = dft.grid.Grids(mol_pert)
    grids_pert.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids_pert.build(sort_grids=False)
    return mol_pert, grids_pert

## 1. PySCF 二阶解析格点导数参考

由于 PySCF 没有实现二阶解析格点导数，我们以一阶解析导数 `hessian.rks.get_dweight_dA` 的二阶中心差分作为参考 `ddw_ref`。**注意**：`get_dweight_dA` 的第一个参数应传入**微扰后**的分子 `mol_pert` (而非未微扰的 `mol`)，否则格点 `grids_pert` (已按微扰坐标重建) 与分子坐标不一致，参考值会出错。

In [3]:
mol, grids = perturb_mol_grids(A=0, t=0, sgn=1, delta=0.0)

nonpad_mask = grids.atm_idx != -1
quadrature_weights = grids.quadrature_weights[nonpad_mask]
grid_coords = grids.coords[nonpad_mask]
atm_coords = mol.atom_coords()
atm_indices = grids.atm_idx[nonpad_mask]

natm = atm_coords.shape[0]
ngrids = grid_coords.shape[0]
becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
radii_table = np.array([becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)

In [4]:
# ddw_ref = central FD of get_dweight_dA(mol_pert, grids_pert)  (perturbed mol!)
interval = 3e-5
ddw_ndiff = np.zeros((natm, 3, natm, 3, ngrids))
for A in range(natm):
    for t in range(3):
        mol_p, grids_p = perturb_mol_grids(A, t, +1, interval)
        mol_m, grids_m = perturb_mol_grids(A, t, -1, interval)
        dw_plus  = hessian.rks.get_dweight_dA(mol_p, grids_p)[..., nonpad_mask]
        dw_minus = hessian.rks.get_dweight_dA(mol_m, grids_m)[..., nonpad_mask]
        ddw_ndiff[A, t] = (dw_plus - dw_minus) / (2 * interval / data.nist.BOHR)
ddw_ref = 0.5 * (ddw_ndiff + ddw_ndiff.transpose(2, 3, 0, 1, 4))  # symmetrize

## 2. 格点计算编写规则

本 notebook 沿用 `10-4` 的迁移原则 (Python 向量化全张量 → Rust 循环 + SIMD)，但有一处重要放宽：

- **结果张量是 $O(N^3)$ 的**。$\partial^2 w_g / \partial R_{At} \partial R_{Bs}$ 的维度为 $(A, t, B, s, g) = (n_\mathrm{atm}, 3, n_\mathrm{atm}, 3, n_\mathrm{batch})$，这是输出本身、不可压缩。因此 `10-4` 中"不能出现 $O(N^3)$ 内存用量"对**输出张量**不适用。
- **但中间量应尽量压缩**。本实现的中间量上限为：
  - 输出张量 `ddR_Z`, `ddR_Pg`, `ddw`：$(n_\mathrm{atm}, 3, n_\mathrm{atm}, 3, n_\mathrm{batch})$，不可避免；
  - `dR_log_P`：$(n_\mathrm{atm}, n_\mathrm{atm}, 3, n_\mathrm{batch})$，4 维，是交叉项 $\sum_M P_M (\partial\log P_M)^{\otimes 2}$ 所必需的最小中间量 (约为一个 5 维输出的 $1/3$)；
  - per-pair 量：$(3, n_\mathrm{batch})$ / $(3, 3, n_\mathrm{batch})$，小。
- `10-3` 中的 6 维 `ddR_log_P`、`ddR_P` **完全不物化**；二阶对数导数在 per-pair 循环中直接累加进 5 维输出。

## 3. 零阶重新实现

### 3.1 格点无关量

与 `10-4` 完全一致，这里仅列出。

In [5]:
wquad = quadrature_weights
a = radii_table
assert np.allclose(a, -a.T)  # Becke radii adjust table is antisymmetric

atom_dist = scipy.spatial.distance.cdist(atm_coords, atm_coords)
for i in range(natm):
    atom_dist[i, i] = np.inf
# dR_atom_dist_{ABt} = d |R_{AB}| / dR_{At} = (R_A - R_B)/|R_{AB}|
dR_atom_dist = (atm_coords[:, None, :] - atm_coords[None, :, :]) / atom_dist[:, :, None]
assert np.allclose(-dR_atom_dist.swapaxes(0, 1), dR_atom_dist)

### 3.2 switch 函数

$s(\mu) = s_3 \circ \nu(\mu)$，$\nu(\mu)=\mu+a(1-\mu^2)$，$f_3(\nu)=p\circ p\circ p(\nu)$，$p(x)=\tfrac32 x-\tfrac12 x^3$，$s_3(\nu)=\tfrac12(1-f_3)$。计算 $f_3$ (而非最终 $s$)，以保留 $\mu_{MN}\to -\mu_{NM}$ 的反对称性。

In [6]:
def switch_f3(mu, a_factor):
    nu = mu + a_factor * (1 - mu * mu)  # eq (A2)
    f1 = (1.5 - 0.5 * nu * nu) * nu     # eq (19)
    f2 = (1.5 - 0.5 * f1 * f1) * f1
    f3 = (1.5 - 0.5 * f2 * f2) * f2
    return f3

## 4. 一阶重新实现

### 4.1 switch 函数的一阶与二阶 $\nu$-导数

记 $p'(x)=\tfrac32(1-x^2)$，$p''(x)=-3x$，$g_i=p'(f_{i-1})$ ($f_0=\nu$)。则

$$f_3'(\nu)=g_2 g_1 g_0, \qquad f_3''(\nu)=-3\big[f_2 (g_1 g_0)^2 + f_1 g_2 g_0^2 + \nu g_2 g_1\big].$$

注意 $f_3$ 为奇函数故 $f_3'$ 为**偶**、$f_3''$ 为奇。$s(\mu)$ 的导数 (链式)：

$$s'(\mu)=-\tfrac12 f_3'(\nu)\,\nu', \qquad s''(\mu)=-\tfrac12 f_3''(\nu)(\nu')^2 - \tfrac12 f_3'(\nu)\,\nu'', \quad \nu'=1-2a\mu,\ \nu''=-2a.$$

In [7]:
def switch_derivs(mu, a_factor):
    """Return f3, f3'(nu), f3''(nu) where nu = mu + a(1-mu^2)."""
    nu = mu + a_factor * (1 - mu * mu)
    f1 = (1.5 - 0.5 * nu * nu) * nu
    f2 = (1.5 - 0.5 * f1 * f1) * f1
    f3 = (1.5 - 0.5 * f2 * f2) * f2
    g0 = 1.5 * (1 - nu * nu)
    g1 = 1.5 * (1 - f1 * f1)
    g2 = 1.5 * (1 - f2 * f2)
    f3p = g2 * g1 * g0                                   # f3'(nu)  (= 10-4's dnu_f3)
    f3pp = -3.0 * (f2 * (g1 * g0)**2 + f1 * g2 * g0**2 + nu * g2 * g1)   # f3''(nu)
    return f3, f3p, f3pp

# self-check via finite difference of f3(mu)  (h ~ eps^(1/4) for d2)
_mu = np.array([0.1, 0.3, -0.2, 0.5]); _a = 0.15; _h = 1e-4
_fd1 = (switch_f3(_mu + _h, _a) - switch_f3(_mu - _h, _a)) / (2 * _h)
_fd2 = (switch_f3(_mu + _h, _a) - 2 * switch_f3(_mu, _a) + switch_f3(_mu - _h, _a)) / _h**2
_f3, _f3p, _f3pp = switch_derivs(_mu, _a)
_dnu = 1 - 2 * _a * _mu; _ddnu = -2 * _a
assert np.allclose(_fd1, _f3p * _dnu, atol=1e-8)
assert np.allclose(_fd2, _f3pp * _dnu**2 + _f3p * _ddnu, atol=1e-6)
print("switch 1st/2nd derivative self-check OK")

switch 1st/2nd derivative self-check OK


### 4.2 任意 hardness 的 switch 函数导数

`10-4` 给出了任意 hardness 的零阶 `switch_fhardness` 与一阶 `switch_dnu_fhardness` (返回 $f$, $f'=\mathrm{d}f/\mathrm{d}\nu$)。这里补上二阶版本 `switch_ddnu_fhardness` (返回 $f$, $f'$, $f''=\mathrm{d}^2 f/\mathrm{d}\nu^2$)，使任意 hardness 的二阶格点导数成为可能。

$f=p\circ\cdots\circ p(\nu)$ (h 次) 的链式递推 (记 $g=p'(f)$)：

$$f_{k+1}=p(f_k),\quad f_{k+1}'=g_k f_k',\quad f_{k+1}''=-3 f_k (f_k')^2 + g_k f_k'',$$

初值 $f_0=\nu$, $f_0'=1$, $f_0''=0$。注意 $\mathrm{d}g_k/\mathrm{d}\nu=p''(f_k) f_k'=(-3f_k) f_k'$，故 $f''$ 的递推先算 (用旧 $f, f', f''$)、再算 $f'$、最后更新 $f$。

In [8]:
def switch_fhardness(mu, a_factor, hardness):
    nu = mu + a_factor * (1 - mu * mu)  # eq (A2)
    f = nu
    for _ in range(hardness):
        f = (1.5 - 0.5 * f * f) * f     # eq (19)
    return f

def switch_dnu_fhardness(mu, a_factor, hardness):
    nu = mu + a_factor * (1 - mu * mu)
    f = nu; df = 1.0
    for _ in range(hardness):
        df = 1.5 * (1 - f * f) * df      # f' recurrence (uses old f)
        f  = (1.5 - 0.5 * f * f) * f     # f = p(f)
    return f, df

def switch_ddnu_fhardness(mu, a_factor, hardness):
    """Return f, f'(nu), f''(nu) for f = p^hardness(nu), nu = mu + a(1-mu^2)."""
    nu = mu + a_factor * (1 - mu * mu)
    f = nu; df = 1.0; ddf = 0.0
    for _ in range(hardness):
        g   = 1.5 * (1 - f * f)            # p'(f)
        ddf = -3.0 * f * df * df + g * ddf  # f'' recurrence (old f, df, ddf)
        df  = g * df                        # f'  recurrence (old df)
        f   = (1.5 - 0.5 * f * f) * f       # f = p(f) (old f)
    return f, df, ddf

# consistency with the specialized hardness=3 versions
assert np.allclose(switch_fhardness(_mu, _a, 3), switch_f3(_mu, _a))
assert np.allclose(switch_dnu_fhardness(_mu, _a, 3)[1], switch_derivs(_mu, _a)[1])
assert np.allclose(switch_ddnu_fhardness(_mu, _a, 3)[2], switch_derivs(_mu, _a)[2])

# FD self-check of f' and f'' for arbitrary hardness
# (rtol accommodates FD truncation ~h^2*|f'''| which grows with hardness as the
#  switch steepens; a wrong formula gives ~O(1) relative error and is caught.)
_h = 1e-4
for _hard in [3, 4, 5, 6]:
    _fd1 = (switch_fhardness(_mu + _h, _a, _hard) - switch_fhardness(_mu - _h, _a, _hard)) / (2 * _h)
    _fd2 = (switch_fhardness(_mu + _h, _a, _hard) - 2 * switch_fhardness(_mu, _a, _hard)
            + switch_fhardness(_mu - _h, _a, _hard)) / _h**2
    _f, _df, _ddf = switch_ddnu_fhardness(_mu, _a, _hard)
    assert np.allclose(_fd1, _df * _dnu, rtol=1e-3, atol=1e-6), f"hardness={_hard} 1st deriv"
    assert np.allclose(_fd2, _ddf * _dnu**2 + _df * _ddnu, rtol=1e-3, atol=1e-6), f"hardness={_hard} 2nd deriv"
print("arbitrary-hardness switch derivative self-check OK (hardness 3-6)")

arbitrary-hardness switch derivative self-check OK (hardness 3-6)


## 5. 二阶重新实现

### 5.1 记号约定

沿用 `10-3`、`10-4` 记号：$A,B$ 为导数原子 $R_{At},R_{Bs}$；$M,N$ 为 $\mu_{MNg}$、$s_{MNg}$ 的两端点；$A_g$ 为格点所属原子。格点坐标 $r_g$ 暂视为不依赖 $R$ (全导数问题留至最后)。二阶导数记号：`dR_*`/`dmu_*` 一阶、`ddR_*`/`ddmu_*` 二阶。

per-pair 循环对无序对 $(A_{atm}, B_{atm})$ ($B_{atm}<A_{atm}$) 进行，pair 自变量 $\mu=\mu_{A_{atm},B_{atm}}$。约定 (与 `10-4` 一致)：

- `dmu_log_sA` $=\partial\log s_{A_{atm},B_{atm}}/\partial\mu$ (自然，$=+dmu\_log\_s[A_{atm},B_{atm}]$)；
- `dmu_log_sB` $=\partial\log s_{B_{atm},A_{atm}}/\partial\mu$ (按 pair 的 $\mu$)，$=-dmu\_log\_s[B_{atm},A_{atm}]$ (因 $f_3'$ 为偶、$f_3''$ 为奇，一阶 log 导数反号、二阶同号)。

据此，`dR_log_P`、L2 的 `common`/`common_dd` 因子均为**全加号**无符号翻转 (与 `10-4` 的 `common_Z` 一致)。

### 5.2 椭球坐标 $\mu_{ABg}$ 的二阶坐标导数 (per-pair)

对 pair $(A_{atm}, B_{atm})$，记 $f=\Vert r\Vert_{A_{atm},g}-\Vert r\Vert_{B_{atm},g}$，$g=\Vert R\Vert_{A_{atm}B_{atm}}$，$\mu=f/g$。一阶 role 导数 (与 `10-4` 一致)：

$$\partial_{R_{A_{atm}}}\mu=\frac{\bm r_{A_{atm}}-\mu\bm R_{AB}}{g},\quad \partial_{R_{B_{atm}}}\mu=\frac{-\bm r_{B_{atm}}+\mu\bm R_{AB}}{g}.$$

二阶由商法则 $\partial_{ij}(f/g)=\big[f_{ij}g-(f_i g_j+g_i f_j)-f g_{ij}\big]/g^2+2f g_i g_j/g^3$ 给出，三 role 块 (记 $\mathrm{Proj}(\bm v)=I-\bm v\bm v^T$)：

| role | $f_X, g_X$ | $f_{XY}$ | $g_{XY}$ |
|---|---|---|---|
| AA (均 $A_{atm}$) | $\bm r_A,\ \bm R_{AB}$ | $\mathrm{Proj}(\bm r_A)/\Vert r\Vert_A$ | $\mathrm{Proj}(\bm R_{AB})/g$ |
| AB ($A_{atm}$ 后 $B_{atm}$) | $\bm r_A,\ -\bm r_B;\ \bm R_{AB},\ -\bm R_{AB}$ | $0$ | $-\mathrm{Proj}(\bm R_{AB})/g$ |
| BB (均 $B_{atm}$) | $-\bm r_B,\ -\bm R_{AB}$ | $-\mathrm{Proj}(\bm r_B)/\Vert r\Vert_B$ | $\mathrm{Proj}(\bm R_{AB})/g$ |

role BA $=$ role AB 在 $(t,s)$ 转置。

### 5.3 权重 $P_{Mg}$、归一化 $Z_g$ 与格点权重 $w_g$ 的二阶导数

二阶用对数法。对 $\ln P_{Mg}=\sum_{N\neq M}\ln s_{MNg}$：

$$\partial^2\log P_{Mg}=\sum_{N\neq M}\big[w_{MNg}(\partial_A\mu)(\partial_B\mu)+t_{MNg}\,\partial_{AB}\mu\big],$$

其中 $w=$ `ddmu_log_s` (二阶)、$t=$ `dmu_log_s` (一阶)。**注意**：$w\cdot(\partial_A\mu)(\partial_B\mu)$ 项用 $\mu$ 的**一阶 role 导数** `dR_mu_roleA/B` ($=(\bm r-\mu\bm R_{AB})/g$)，而非单位向量 $\bm r$。最后

$$\partial^2 P_{Mg}=P_{Mg}\big(\partial^2\log P_{Mg}+\partial\log P_{Mg}\otimes\partial\log P_{Mg}\big),\quad \partial^2 Z_g=\sum_M\partial^2 P_{Mg}.$$

per-pair 循环中，L2 项 ($\partial^2\log P$，对 $M\in\{A_{atm},B_{atm}\}$ 的 4 个 role 块) 直接累加进 5 维 `ddR_Z`/`ddR_Pg`；交叉项 $\partial\log P\otimes\partial\log P$ (需完整 `dR_log_P`) 用 einsum 一次性补上。

格点权重 $w_g=w_g^{quad} P^g_{A_g}/Z_g$ 的二阶偏导 (商法则，$r_g$ 固定)：

$$\partial^2_{AB} q_g=\frac{\partial_{AB}P_{A_g}-(\partial_B q)(\partial_A Z)-q\,\partial_{AB}Z}{Z_g}-\frac{(\partial_A q)(\partial_B Z)}{Z_g},\quad w_g^{quad}\cdot\partial^2_{AB}q_g,$$

再由平移不变性 $\sum_A\partial_{AB}w_g=0$ 补 $A=A_g$、$B=A_g$ 块 (求和须排除 $A_g$，角点 $(A_g,A_g)$ 为双重求和)。

### 5.4 完整 per-batch 实现

单个 batch 循环同时给出 $w_g$ (零阶)、$\partial w_g/\partial R_{At}$ (一阶 `dw`)、$\partial^2 w_g/\partial R_{At}\partial R_{Bs}$ (二阶 `ddw`)：

- **Pass 1**：连乘积 $P_{Mg}=\prod_{N\neq M}s_{MNg}$；
- **Pass 2** (per-pair)：累加 `dR_log_P` (4 维)、`dR_Z`/`dR_Pg` (一阶)、L2 `ddR_Z`/`ddR_Pg` (二阶，5 维)；
- 循环后：交叉项 einsum、商法则、平移不变性修正。

In [9]:
INVTOL = 1e-14
eye3 = np.eye(3)

def proj_v(v):  # v (3, nbatch) -> (3, 3, nbatch) = I - v v^T
    return eye3[:, :, None] - v[:, None, :] * v[None, :, :]

w = np.zeros(ngrids)
dw = np.zeros((natm, 3, ngrids))
ddw = np.zeros((natm, 3, natm, 3, ngrids))

nbatch_grid = 256
for g0 in range(0, ngrids, nbatch_grid):
    # ---- handle batch ---------------------------------------------------
    g1 = min(g0 + nbatch_grid, ngrids)
    nbatch = g1 - g0
    batch_coords = grid_coords[g0:g1]
    batch_dist = scipy.spatial.distance.cdist(atm_coords, batch_coords)        # (natm, nbatch)
    batch_atm_indices = atm_indices[g0:g1]
    # unit vectors r_A = (R_A - r_g)/|r_A|  -> (3, nbatch)
    dR_batch_dist = (atm_coords[:, :, None] - batch_coords.T[None, :, :]) / batch_dist[:, None, :]

    # ---- Pass 1: zero-order P_{Mg} = prod_{N!=M} s_{MNg} ----------------
    P = np.ones((natm, nbatch))
    for Aatm in range(natm):
        for Batm in range(Aatm):
            mu = (batch_dist[Aatm] - batch_dist[Batm]) / atom_dist[Aatm, Batm]
            f3 = switch_f3(mu, a[Aatm, Batm])
            P[Aatm] *= 0.5 * (1 - f3)
            P[Batm] *= 0.5 * (1 + f3)
    Z = P.sum(axis=0)
    Pg = P[batch_atm_indices, np.arange(nbatch)]
    w[g0:g1] = wquad[g0:g1] * Pg / Z

    # ---- Pass 2: per-pair accumulation (1st + 2nd order) ----------------
    # dR_log_P (M, A, t, g): 4D, minimal necessary for the cross term
    dR_log_P = np.zeros((natm, natm, 3, nbatch))
    dR_Z  = np.zeros((natm, 3, nbatch))
    dR_Pg = np.zeros((natm, 3, nbatch))
    ddR_Z  = np.zeros((natm, 3, natm, 3, nbatch))   # 5D output (L2 part)
    ddR_Pg = np.zeros((natm, 3, natm, 3, nbatch))   # 5D output (L2 part)

    for Aatm in range(natm):
        for Batm in range(Aatm):
            P_A = P[Aatm]; P_B = P[Batm]
            af = a[Aatm, Batm]
            rA = dR_batch_dist[Aatm]                # (3, nbatch) unit vec R_A - r_g
            rB = dR_batch_dist[Batm]
            Uvec = dR_atom_dist[Aatm, Batm]         # (3,) unit vec R_A - R_B
            g_ab = atom_dist[Aatm, Batm]             # scalar |R_AB|
            dA = batch_dist[Aatm]; dB = batch_dist[Batm]
            f = dA - dB                              # (nbatch,)
            mu = f / g_ab                            # (nbatch,)

            # 1st role derivatives of mu_{AB}
            dR_mu_roleA = (  rA - mu[None, :] * Uvec[:, None]) / g_ab   # d mu/dR_A  (3, nbatch)
            dR_mu_roleB = (- rB + mu[None, :] * Uvec[:, None]) / g_ab   # d mu/dR_B  (3, nbatch)

            # switch value + 1st/2nd nu-derivs -> s(mu) and its 1st/2nd mu-derivs
            f3, f3p, f3pp = switch_derivs(mu, af)
            sA = 0.5 * (1 - f3)      # s_{AB}
            sB = 0.5 * (1 + f3)      # s_{BA} = 1 - sA
            dmu_nu = 1 - 2 * af * mu
            ddmu_nu = -2 * af
            dmu_sA  = -0.5 * f3p * dmu_nu                 # ds_{AB}/dmu
            dmu_sB  =  0.5 * f3p * dmu_nu                 # ds_{BA}/dmu  (= -dmu_sA)
            ddmu_sA = -0.5 * f3pp * dmu_nu**2 - 0.5 * f3p * ddmu_nu
            ddmu_sB =  0.5 * f3pp * dmu_nu**2 + 0.5 * f3p * ddmu_nu   # = -ddmu_sA

            # log-derivatives wrt mu (regularize small s: s_safe = 1.0 where |s|<=INVTOL,
            # i.e. dmu_log_s = dmu_s raw there; NOT the 1e-14 floor of 10-4, which blows
            # up the 2nd-order L2/cross terms).
            maskA = sA > INVTOL; maskB = sB > INVTOL
            sA_safe = np.where(maskA, sA, 1.0)
            sB_safe = np.where(maskB, sB, 1.0)
            dmu_log_sA = dmu_sA / sA_safe
            dmu_log_sB = dmu_sB / sB_safe
            ddmu_log_sA = np.where(maskA, ddmu_sA / sA_safe, 0.0) - dmu_log_sA**2
            ddmu_log_sB = np.where(maskB, ddmu_sB / sB_safe, 0.0) - dmu_log_sB**2

            # 2nd role derivatives of mu_{AB} (3 blocks), via quotient rule (10-3 S5)
            Ub  = np.broadcast_to(Uvec[:, None], (3, nbatch))       # R_AB unit vec over batch
            PrA = proj_v(rA) / dA[None, None, :]                    # Proj(r_A)/|r_A|   (3,3,nbatch)
            PrB = proj_v(rB) / dB[None, None, :]                    # Proj(r_B)/|r_B|
            PU  = proj_v(Ub) / g_ab                                 # Proj(R_AB)/|R_AB|
            zero_ts = np.zeros((3, 3, 1))
            def d2mu(fX, fY, fXY, gX, gY, gXY):
                ofg = fX[:, None, :] * gY[None, :, :] + gX[:, None, :] * fY[None, :, :]
                ogg = gX[:, None, :] * gY[None, :, :]
                return (fXY * g_ab - ofg - f[None, None, :] * gXY) / g_ab**2 \
                       + 2 * f[None, None, :] * ogg / g_ab**3
            ddR_mu_roleAA = d2mu( rA,  rA,  PrA,  Ub,  Ub,  PU)
            ddR_mu_roleAB = d2mu( rA, -rB,  zero_ts, Ub, -Ub, -PU)
            ddR_mu_roleBB = d2mu(-rB, -rB, -PrB, -Ub, -Ub,  PU)
            ddR_mu_roleBA = ddR_mu_roleAB.swapaxes(0, 1)

            # ---- accumulate dR_log_P (4D), dR_Z, dR_Pg (1st order) ----
            # convention: dmu_log_sA = +nat[A,B], dmu_log_sB = -nat[B,A]; all-plus below.
            dR_log_P[Aatm, Aatm] += dmu_log_sA[None, :] * dR_mu_roleA   # M=A, role A
            dR_log_P[Aatm, Batm] += dmu_log_sA[None, :] * dR_mu_roleB   # M=A, role B
            dR_log_P[Batm, Aatm] += dmu_log_sB[None, :] * dR_mu_roleA   # M=B, role B (a=N=Aatm)
            dR_log_P[Batm, Batm] += dmu_log_sB[None, :] * dR_mu_roleB   # M=B, role A (a=M=Batm)
            common_Z = P_A * dmu_log_sA + P_B * dmu_log_sB              # = 10-4's common_Z
            dR_Z[Aatm] += common_Z[None, :] * dR_mu_roleA
            dR_Z[Batm] += common_Z[None, :] * dR_mu_roleB
            maskAg = batch_atm_indices == Aatm
            maskBg = batch_atm_indices == Batm
            common_Pg = (np.where(maskAg, P_A * dmu_log_sA, 0.0)
                       + np.where(maskBg, P_B * dmu_log_sB, 0.0))
            dR_Pg[Aatm] += common_Pg[None, :] * dR_mu_roleA
            dR_Pg[Batm] += common_Pg[None, :] * dR_mu_roleB

            # ---- accumulate L2 (2nd log-deriv) into ddR_Z, ddR_Pg (5D) ----
            # ddR_Z = sum_M P_M * ddR_log_P; the w*(d_A mu)(d_B mu) term uses the
            # FIRST role derivatives dR_mu_roleA/B (NOT the unit vectors rA/rB).
            common_dd = P_A * ddmu_log_sA + P_B * ddmu_log_sB
            def outer(rX, rY):  # (3,3,nbatch)
                return rX[:, None, :] * rY[None, :, :]
            ddR_Z[Aatm, :, Aatm, :] += common_dd[None, None, :] * outer(dR_mu_roleA, dR_mu_roleA) \
                                    + common_Z [None, None, :] * ddR_mu_roleAA
            ddR_Z[Aatm, :, Batm, :] += common_dd[None, None, :] * outer(dR_mu_roleA, dR_mu_roleB) \
                                    + common_Z [None, None, :] * ddR_mu_roleAB
            ddR_Z[Batm, :, Aatm, :] += common_dd[None, None, :] * outer(dR_mu_roleB, dR_mu_roleA) \
                                    + common_Z [None, None, :] * ddR_mu_roleBA
            ddR_Z[Batm, :, Batm, :] += common_dd[None, None, :] * outer(dR_mu_roleB, dR_mu_roleB) \
                                    + common_Z [None, None, :] * ddR_mu_roleBB
            # ddR_Pg: pick M = A_g; M=A block on maskAg, M=B block on maskBg
            coef_A = np.where(maskAg, P_A, 0.0); coef_B = np.where(maskBg, P_B, 0.0)
            c1_Pg  = coef_A * dmu_log_sA  + coef_B * dmu_log_sB
            cdd_Pg = coef_A * ddmu_log_sA + coef_B * ddmu_log_sB
            ddR_Pg[Aatm, :, Aatm, :] += cdd_Pg[None, None, :] * outer(dR_mu_roleA, dR_mu_roleA) \
                                     + c1_Pg [None, None, :] * ddR_mu_roleAA
            ddR_Pg[Aatm, :, Batm, :] += cdd_Pg[None, None, :] * outer(dR_mu_roleA, dR_mu_roleB) \
                                     + c1_Pg [None, None, :] * ddR_mu_roleAB
            ddR_Pg[Batm, :, Aatm, :] += cdd_Pg[None, None, :] * outer(dR_mu_roleB, dR_mu_roleA) \
                                     + c1_Pg [None, None, :] * ddR_mu_roleBA
            ddR_Pg[Batm, :, Batm, :] += cdd_Pg[None, None, :] * outer(dR_mu_roleB, dR_mu_roleB) \
                                     + c1_Pg [None, None, :] * ddR_mu_roleBB

    # ---- cross term: sum_M P_M (dR_log_P_M)(dR_log_P_M) -----------------
    ddR_Z  += np.einsum("Mg,MAtg,MBsg->AtBsg", P, dR_log_P, dR_log_P)
    # gather M = A_g:  dlog_Ag[a, t, g] = dR_log_P[atm_indices[g], a, t, g]
    dlog_Ag = dR_log_P.transpose(0, 3, 1, 2)[batch_atm_indices, np.arange(nbatch)].transpose(1, 2, 0)  # (A, t, g)
    P_Ag = P[batch_atm_indices, np.arange(nbatch)]                    # (g,)
    ddR_Pg += np.einsum("g,atg,bsg->atbsg", P_Ag, dlog_Ag, dlog_Ag)

    # ---- quotient rule for dw (1st order, r_g fixed) -------------------
    dw_batch = wquad[g0:g1][None, None, :] * (dR_Pg / Z[None, None, :]
        - Pg[None, None, :] / Z[None, None, :]**2 * dR_Z)
    for g in range(nbatch):
        Ag = batch_atm_indices[g]
        sum_excl = dw_batch[:, :, g].sum(axis=0) - dw_batch[Ag, :, g]
        dw_batch[Ag, :, g] = -sum_excl
    dw[:, :, g0:g1] = dw_batch

    # ---- quotient rule for ddw (2nd order, r_g fixed) ------------------
    q = Pg / Z
    dq = (dR_Pg - q[None, None, :] * dR_Z) / Z[None, None, :]         # (A, t, g)
    term1 = np.einsum("Bsg,Atg->AtBsg", dq, dR_Z)                     # (dq_B)(dZ_A)
    term2 = np.einsum("Atg,Bsg->AtBsg", dq, dR_Z)                     # (dq_A)(dZ_B)
    d2q = (ddR_Pg - term1 - q[None, None, None, None, :] * ddR_Z) / Z[None, None, None, None, :] \
          - term2 / Z[None, None, None, None, :]
    ddw_partial = wquad[g0:g1][None, None, None, None, :] * d2q       # (A, t, B, s, g)

    # translation-invariance fix (exclude A_g from both sums; corner = double sum)
    ddw_batch = ddw_partial.copy()
    for g in range(nbatch):
        Ag = batch_atm_indices[g]
        sl = ddw_partial[:, :, :, :, g]
        sumB_ne = sl.sum(axis=2) - sl[:, :, Ag]    # sum over B != A_g
        sumA_ne = sl.sum(axis=0) - sl[Ag]          # sum over A != A_g
        ddw_batch[Ag, :, :, :, g] = -sumA_ne
        ddw_batch[:, :, Ag, :, g] = -sumB_ne
        ddw_batch[Ag, :, Ag, :, g] = sumB_ne.sum(axis=0) - sumB_ne[Ag]
    ddw[:, :, :, :, g0:g1] = ddw_batch

## 6. 验证

将解析二阶导数 `ddw` 与数值参考 `ddw_ref` 比较。`ddw_ref` 为 PySCF `get_dweight_dA` (一阶解析) 的二阶中心差分，误差主要来自有限差分截断 $\sim h^2$；远离原子核、$|s_{AB}|$ 较小的格点处截断误差更显著。同时检验 `ddw` 的 $(A,t)\leftrightarrow(B,s)$ 对称性，以及 $w$、$\partial w_g/\partial R_{At}$ 的一致。

In [10]:
print(f"w   match weights: {np.allclose(w, grids.weights[nonpad_mask])}")
dw_ref = hessian.rks.get_dweight_dA(mol, grids)[..., nonpad_mask]
print(f"dw  match: median={np.median(np.abs(dw - dw_ref)):.4e}  allclose={np.allclose(dw, dw_ref)}")

diff = np.abs(ddw - ddw_ref)
print(f"ddw median|diff| = {np.median(diff):.4e}")
print(f"ddw max|diff|    = {np.max(diff):.4e}  (far grids: FD truncation ~h^2 in ddw_ref)")
print(f"ddw 99.9p|diff|  = {np.percentile(diff, 99.9):.4e}")
print(f"symmetry max|ddw - ddw.T(A<->B)| = {np.max(np.abs(ddw - ddw.transpose(2, 3, 0, 1, 4))):.4e}")
print(f"max|ddw|={np.max(np.abs(ddw)):.4e}  max|ddw_ref|={np.max(np.abs(ddw_ref)):.4e}")
print(f"\nrtol=1e-4, atol=1e-6: {np.allclose(ddw, ddw_ref, rtol=1e-4, atol=1e-6)}")

# spot-check a few (A, t, B, s) blocks
for (A, a, B, b, lab) in [(0,0,0,0,"0x0x"), (0,0,0,1,"0x0y"), (0,0,1,0,"0x1x"), (0,0,1,1,"0x1y")]:
    d = np.abs(ddw[A, a, B, b] - ddw_ref[A, a, B, b])
    print(f"  {lab}: max|ddw|={np.max(np.abs(ddw[A,a,B,b])):.4e} "
          f"median|diff|={np.median(d):.4e} max|diff|={np.max(d):.4e}")

w   match weights: True
dw  match: median=8.4703e-22  allclose=True
ddw median|diff| = 1.3685e-14
ddw max|diff|    = 1.5023e-06  (far grids: FD truncation ~h^2 in ddw_ref)
ddw 99.9p|diff|  = 2.5387e-08
symmetry max|ddw - ddw.T(A<->B)| = 1.4211e-14
max|ddw|=1.1543e+02  max|ddw_ref|=1.1543e+02

rtol=1e-4, atol=1e-6: True
  0x0x: max|ddw|=8.6315e+01 median|diff|=6.0545e-13 max|diff|=1.0921e-06
  0x0y: max|ddw|=3.6751e+01 median|diff|=3.0778e-13 max|diff|=2.5697e-07
  0x1x: max|ddw|=1.0906e+01 median|diff|=5.9025e-14 max|diff|=3.0129e-08
  0x1y: max|ddw|=3.0367e+01 median|diff|=3.6930e-14 max|diff|=1.3881e-07
